<a href="https://colab.research.google.com/github/anikaumaselvan/academic_projects/blob/main/Anika.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shoe Store Data Pipeline
## Objective: Increase Total Online Sales and Average Transaction Value




In [ ]:

!pip install pymysql pandas numpy scikit-learn plotly ipywidgets sqlalchemy --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import pymysql
import warnings
warnings.filterwarnings('ignore')

from sqlalchemy import create_engine, text
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.engine.url import URL

db_url = URL.create(
    drivername="mysql+pymysql",
   host= '23.22.155.107',
   port=3306,
   username= 'test1',
  password= 'Test1234#',
  database="team_8_sneaker_marketplace"
)
db_engine = create_engine(db_url)


In [ ]:
extract_query = """
    SELECT
        oi.order_item_id,
        oi.order_id,
        o.user_id,
        CONCAT(u.first_name, ' ', u.last_name) AS user_name,
        oi.product_id,
        p.brand,
        p.model,
        oi.quantity,
        oi.unit_price,
        oi.timestamp
    FROM order_item oi
    JOIN `order` o  ON oi.order_id   = o.order_id
    JOIN user u     ON o.user_id     = u.user_id
    JOIN product p  ON oi.product_id = p.product_id
    WHERE LOWER(o.order_status) NOT IN ('cancelled')
"""
raw_df = pd.read_sql(extract_query, db_engine)

raw_df.head()

,order_item_id,order_id,user_id,user_name,product_id,brand,model,quantity,unit_price,timestamp
0,10361,5010,1144,Layla Ross,20000,Nike,Jordan 1 Retro,2,184.49,2026-04-01 04:00:00
1,10226,5067,1014,Owen Reed,20000,Nike,Jordan 1 Retro,1,178.99,2026-03-26 11:00:00
2,10370,5190,1062,Owen Cooper,20001,Nike,Jordan 1 Retro,3,175.50,2026-03-28 20:00:00
3,10093,5166,1030,Vivian Gray,20001,Nike,Jordan 1 Retro,3,173.79,2026-03-12 23:00:00
4,10074,5145,1067,Micah Torres,20001,Nike,Jordan 1 Retro,4,186.93,2026-03-28 14:00:00


In [ ]:
etl_df = raw_df.copy()



etl_df['line_total'] = etl_df['quantity'] * etl_df['unit_price']


brand_median = etl_df.groupby('brand')['unit_price'].transform('median')
etl_df['unit_price'] = etl_df['unit_price'].fillna(brand_median)
etl_df['line_total']  = etl_df['line_total'].fillna(etl_df['quantity'] * etl_df['unit_price'])


before_dedup = len(etl_df)
etl_df = etl_df.drop_duplicates(subset=['order_id', 'product_id', 'timestamp', 'unit_price', 'quantity'])



price_mean = etl_df['unit_price'].mean()
price_std  = etl_df['unit_price'].std()
upper_cap  = price_mean + 3 * price_std
outliers   = etl_df[etl_df['unit_price'] > upper_cap]
etl_df     = etl_df[etl_df['unit_price'] <= upper_cap]



scaler = MinMaxScaler()
etl_df['unit_price_normalized'] = scaler.fit_transform(etl_df[['unit_price']])


etl_df['user_name'] = etl_df['user_name'].str.title()

etl_df[['order_item_id','user_name','brand','model','quantity','unit_price','unit_price_normalized','line_total']].head(10)

,order_item_id,user_name,brand,model,quantity,unit_price,unit_price_normalized,line_total
0,10361,Layla Ross,Nike,Jordan 1 Retro,2,184.49,0.720790,368.98
1,10226,Owen Reed,Nike,Jordan 1 Retro,1,178.99,0.691259,178.99
2,10370,Owen Cooper,Nike,Jordan 1 Retro,3,175.50,0.672519,526.50
3,10093,Vivian Gray,Nike,Jordan 1 Retro,3,173.79,0.663338,521.37
4,10074,Micah Torres,Nike,Jordan 1 Retro,4,186.93,0.733892,747.72
5,10255,Mila Ramirez,Nike,Jordan 1 Retro,4,174.00,0.664465,696.00
6,10000,Stella Price,Nike,Jordan 1 Retro,1,176.72,0.679070,176.72
7,10335,Isaiah Rivera,Nike,Jordan 1 Retro,2,175.86,0.674452,351.72
8,10083,Hannah Peterson,Nike,Jordan 1 Retro,4,175.99,0.675150,703.96
9,10033,Chloe Brooks,Nike,Jordan 1 Retro,3,181.89,0.706830,545.67


In [ ]:
with db_engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print(result.fetchone())

(1,)


In [ ]:
with db_engine.connect() as conn:
    conn.rollback()
etl_df[['order_item_id','order_id','user_id','user_name','product_id',
        'brand','model','quantity','unit_price','unit_price_normalized',
        'line_total','timestamp']].to_sql(
    'order_item_cleaned', db_engine,
    if_exists='replace', index=False
)

328

In [ ]:

reviews_raw = pd.read_sql("""
    SELECT pr.review_id, pr.user_id, pr.product_id,
           pr.rating, pr.review_text, pr.timestamp
    FROM product_review pr
""", db_engine)

# After reading reviews_raw:
reviews_clean = reviews_raw.copy()

# Drop NULL ratings
reviews_clean = reviews_clean.dropna(subset=['rating'])

# Drop out-of-range ratings (valid range is 1–5)
reviews_clean = reviews_clean[reviews_clean['rating'].between(1, 5)]

# Drop empty review texts
reviews_clean = reviews_clean[reviews_clean['review_text'].notna()]
reviews_clean = reviews_clean[reviews_clean['review_text'].str.strip() != '']

# Use reviews_clean instead of reviews_raw for staging
reviews_clean.to_sql('review_staging', db_engine, if_exists='replace', index=False)

reviews_raw.head()

,review_id,user_id,product_id,rating,review_text,timestamp
0,80000,30175,20049,2.0,Comfortable from day one.,2026-01-17 11:00:00
1,80001,30177,20121,NaN,Good daily shoe but not for running.,2026-03-22 13:00:00
2,80002,30187,20036,1.0,Limited edition worth every penny.,2025-08-16 10:00:00
3,80003,30183,20102,5.0,Delivery was late but shoes are amazing.,2025-09-11 03:00:00
4,80004,30266,20018,2.0,Limited edition worth every penny.,2025-10-24 07:00:00


In [ ]:
with db_engine.connect() as conn:
    # Removed: CREATE TABLE IF NOT EXISTS statement
    # Removed: conn.commit()
    pass

reviews_raw.to_sql('review_staging', db_engine, if_exists='replace', index=False)

300

In [ ]:
with db_engine.connect() as conn:

    conn.execute(text("TRUNCATE TABLE brand_rating_summary"))
    conn.commit()
    conn.execute(text("""
        INSERT INTO brand_rating_summary (brand, avg_rating, review_count)
        SELECT
            CASE
                WHEN LOWER(p.brand) = 'adidas' THEN 'Adidas'
                WHEN LOWER(p.brand) = 'nike' THEN 'Nike'
                WHEN LOWER(p.brand) = 'puma' THEN 'Puma'
                WHEN LOWER(p.brand) = 'new balance' THEN 'New Balance'
                WHEN LOWER(p.brand) = 'reebok' THEN 'Reebok'
                WHEN LOWER(p.brand) = 'on running' THEN 'On Running'
                WHEN LOWER(p.brand) = 'salomon' THEN 'Salomon'
                WHEN LOWER(p.brand) = 'skechers' THEN 'Skechers'
                WHEN LOWER(p.brand) = 'hoka' THEN 'HOKA'
                WHEN LOWER(p.brand) = 'converse' THEN 'Converse'
                WHEN LOWER(p.brand) = 'asics' THEN 'Asics'
                WHEN LOWER(p.brand) = 'under armour' THEN 'Under Armour'
                WHEN LOWER(p.brand) = 'vans' THEN 'VANS'
                ELSE p.brand
            END AS brand_standardized,
            ROUND(AVG(rs.rating), 2) AS avg_rating,
            COUNT(*)                 AS review_count
        FROM review_staging rs
        JOIN product p ON rs.product_id = p.product_id
        GROUP BY brand_standardized
    """))
    conn.commit()

brand_summary = pd.read_sql('SELECT * FROM brand_rating_summary', db_engine)

brand_summary

,brand,avg_rating,review_count
0,Nike,2.97,76
1,Reebok,3.33,20
2,Puma,3.33,26
3,VANS,3.45,14
4,New Balance,2.72,32
5,Asics,3.20,12
6,Adidas,3.02,47
7,On Running,3.50,13
8,Converse,3.00,20
9,Under Armour,2.45,12


Google. (n.d.). Gemini. Google. https://gemini.google.com/ I used Artificial Intelligience for checking my code and for help with the brand rating summary table as I was having issues with my SQL statements. I also used gemini to help reorganize my previous notebook as we shifted to using a unified data insert notebook upstream with preprocessing and cleaning, so a lot of my insert statements had to be removed and worked around.

In [ ]:

interaction_df = pd.read_sql("""
    SELECT user_id, brand, SUM(line_total) as total_spend
    FROM order_item_cleaned
    GROUP BY user_id, brand
""", db_engine)

interaction_matrix = interaction_df.pivot_table(
    index='user_id', columns='brand', values='total_spend', fill_value=0
)
print('User-brand interaction matrix:')
print(interaction_matrix)

User-brand interaction matrix:
brand    Adidas   Asics  Converse    Hoka  New Balance     Nike  On Running  \
user_id                                                                       
1000       0.00    0.00      0.00  150.68       123.35   507.63         0.0   
1001       0.00  470.49      0.00    0.00         0.00   281.97         0.0   
1002       0.00    0.00     71.52    0.00       115.85   233.74         0.0   
1004       0.00  143.30      0.00    0.00         0.00     0.00         0.0   
1005     472.14    0.00      0.00    0.00         0.00   522.18         0.0   
...         ...     ...       ...     ...          ...      ...         ...   
1144     450.36  149.60    144.16    0.00       284.64  1067.83         0.0   
1145     148.86    0.00      0.00    0.00       484.04   281.49         0.0   
1146     150.42    0.00      0.00    0.00         0.00     0.00         0.0   
1147      91.23  153.81      0.00    0.00         0.00     0.00         0.0   
1148       0.00    0.

In [ ]:



matrix_values = interaction_matrix.values
row_sums = matrix_values.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # avoid division by zero
matrix_norm = matrix_values / row_sums


sim_matrix = cosine_similarity(matrix_norm)
sim_df = pd.DataFrame(
    sim_matrix,
    index=interaction_matrix.index,
    columns=interaction_matrix.index
)
print('Cosine similarity matrix (users):')
print(sim_df.round(2))

# recommendations bundle for users


def recommend_bundle(user_id, top_n_users=2, top_n_brands=2):
    if user_id not in sim_df.index:
        return []

    # Find top similar users (excluding self)
    similar_users = (
        sim_df[user_id]
        .drop(user_id)
        .nlargest(top_n_users)
        .index.tolist()
    )


    user_brands = set(
        interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index
    )


    rec_scores = {}
    for su in similar_users:
        for brand in interaction_matrix.columns:
            if brand not in user_brands and interaction_matrix.loc[su, brand] > 0:
                rec_scores[brand] = rec_scores.get(brand, 0) + interaction_matrix.loc[su, brand]


    sorted_recs = sorted(rec_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_recs[:top_n_brands]


all_recs = []
for uid in interaction_matrix.index:
    recs = recommend_bundle(uid)
    for brand, score in recs:
        all_recs.append({'user_id': uid, 'recommended_brand': brand, 'score': round(score, 2)})

recs_df = pd.DataFrame(all_recs)
print('Recommendations generated:')
print(recs_df)

Cosine similarity matrix (users):
user_id  1000  1001  1002  1004  1005  1007  1012  1013  1014  1015  ...  \
user_id                                                              ...   
1000     1.00  0.48  0.90  0.00  0.67  0.00  0.77  0.00  0.73   0.0  ...   
1001     0.48  1.00  0.44  0.86  0.37  0.00  0.57  0.86  0.40   0.0  ...   
1002     0.90  0.44  1.00  0.00  0.62  0.00  0.71  0.00  0.68   0.0  ...   
1004     0.00  0.86  0.00  1.00  0.00  0.00  0.21  1.00  0.00   0.0  ...   
1005     0.67  0.37  0.62  0.00  1.00  0.00  0.90  0.00  0.56   0.0  ...   
...       ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
1144     0.70  0.45  0.71  0.10  0.70  0.00  0.74  0.10  0.55   0.0  ...   
1145     0.62  0.24  0.75  0.00  0.49  0.13  0.49  0.00  0.44   0.0  ...   
1146     0.00  0.00  0.00  0.00  0.65  0.00  0.55  0.00  0.00   0.0  ...   
1147     0.00  0.74  0.00  0.86  0.33  0.00  0.46  0.86  0.00   0.0  ...   
1148     0.71  0.39  0.66  0.00  0.71  0.00  0.58  0.0

Google. (n.d.). Gemini. Google. https://gemini.google.com/ I used the Gemini built in Colab to utilize my interaction matrix in order to make accurate recommendation bundles. The algorithm uses cosine similarity within the matrix to make future recommendation bundles that are accurate to what the user typically purchases based on other customer purchase history.

First the recommend bundle identified users that were most similar to the given user id using cosine similarity matrix, and then iterated through top similar users and the brands they purchase. For brands that the current user is yet to purchase but the similar user has purchased, it creates a recommendation score that is based on how much a similar user spends on that brand. Overall the bundle returns the top recommendations (the brands with the highest accumulated scores) that a user hasn't yet purchased.

In [ ]:
# create recommendations table
with db_engine.connect() as conn:

    pass

if not recs_df.empty:
    recs_df.to_sql('user_recommendations', db_engine, if_exists='replace', index=False)




# Verify by reading back
verify = pd.read_sql('SELECT * FROM user_recommendations', db_engine)
verify

,user_id,recommended_brand,score
0,1000,Converse,308.22
1,1000,Salomon,180.91
2,1002,Salomon,180.91
3,1002,Asics,144.43
4,1005,Skechers,50.25
...,...,...,...
83,1140,Reebok,82.12
84,1143,Nike,435.90
85,1143,Vans,208.41
86,1148,New Balance,176.28


## Interactive Dashboard

This interactive dashboard uses Plotly dropdown buttons instead of ipywidgets, so it should render more reliably in Colab and Jupyter. The dashboard supports the objective of increasing online sales and average transaction value by showing revenue by brand, average transaction value over time, brand ratings, and recommendation strength by brand.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML


try:
    import google.colab  # type: ignore
    pio.renderers.default = "colab"
except Exception:
    pio.renderers.default = "notebook_connected"


cleaned = pd.read_sql('SELECT * FROM order_item_cleaned', db_engine)
brand_stats = pd.read_sql('SELECT * FROM brand_rating_summary', db_engine)

try:
    rec_data = pd.read_sql('SELECT * FROM user_recommendations', db_engine)
except Exception:
    rec_data = pd.DataFrame(columns=['user_id', 'recommended_brand', 'score'])


cleaned['timestamp'] = pd.to_datetime(cleaned['timestamp'], errors='coerce')
cleaned = cleaned.dropna(subset=['timestamp', 'brand', 'line_total'])
cleaned['week'] = cleaned['timestamp'].dt.to_period('W').astype(str)

cleaned['line_total'] = pd.to_numeric(cleaned['line_total'], errors='coerce').fillna(0)
brand_stats['avg_rating'] = pd.to_numeric(brand_stats['avg_rating'], errors='coerce')

if not rec_data.empty and 'score' in rec_data.columns:
    rec_data['score'] = pd.to_numeric(rec_data['score'], errors='coerce').fillna(0)


revenue_by_brand = (
    cleaned.groupby('brand', as_index=False)
    .agg(revenue=('line_total', 'sum'), orders=('order_id', 'nunique'))
    .sort_values('revenue', ascending=False)
)

atv_by_week = (
    cleaned.groupby('week', as_index=False)
    .agg(avg_transaction_value=('line_total', 'mean'), revenue=('line_total', 'sum'))
    .sort_values('week')
)

brand_ratings = (
    brand_stats[['brand', 'avg_rating', 'review_count']]
    .dropna(subset=['brand'])
    .sort_values('avg_rating', ascending=False)
)

if not rec_data.empty and 'recommended_brand' in rec_data.columns:
    rec_scores = (
        rec_data.groupby('recommended_brand', as_index=False)
        .agg(recommendation_score=('score', 'sum'))
        .sort_values('recommendation_score', ascending=False)
    )
else:
    rec_scores = pd.DataFrame({'recommended_brand': [], 'recommendation_score': []})


fig = go.Figure()

# 1. Revenue by brand
fig.add_trace(go.Bar(
    x=revenue_by_brand['brand'],
    y=revenue_by_brand['revenue'],
    name='Revenue by Brand',
    text=revenue_by_brand['revenue'].round(2),
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Revenue: $%{y:,.2f}<extra></extra>',
    visible=True
))

# 2. Average transaction value over time
fig.add_trace(go.Scatter(
    x=atv_by_week['week'],
    y=atv_by_week['avg_transaction_value'],
    name='Average Transaction Value Over Time',
    mode='lines+markers',
    hovertemplate='<b>%{x}</b><br>Avg Transaction Value: $%{y:,.2f}<extra></extra>',
    visible=False
))

# 3. Brand rating
fig.add_trace(go.Bar(
    x=brand_ratings['brand'],
    y=brand_ratings['avg_rating'],
    name='Average Brand Rating',
    text=brand_ratings['avg_rating'].round(2),
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Avg Rating: %{y:.2f}<extra></extra>',
    visible=False
))

# 4. Recommendation scores
fig.add_trace(go.Bar(
    x=rec_scores['recommended_brand'],
    y=rec_scores['recommendation_score'],
    name='Recommendation Score by Brand',
    text=rec_scores['recommendation_score'].round(2),
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Recommendation Score: %{y:,.2f}<extra></extra>',
    visible=False
))

buttons = [
    dict(
        label='Revenue by Brand',
        method='update',
        args=[
            {'visible': [True, False, False, False]},
            {'title': 'Revenue by Brand', 'yaxis': {'title': 'Revenue ($)'}, 'xaxis': {'title': 'Brand'}}
        ]
    ),
    dict(
        label='Average Transaction Value Over Time',
        method='update',
        args=[
            {'visible': [False, True, False, False]},
            {'title': 'Average Transaction Value Over Time', 'yaxis': {'title': 'Average Transaction Value ($)'}, 'xaxis': {'title': 'Week', 'type': 'category'}}
        ]
    ),
    dict(
        label='Average Brand Rating',
        method='update',
        args=[
            {'visible': [False, False, True, False]},
            {'title': 'Average Brand Rating', 'yaxis': {'title': 'Average Rating', 'range': [0, 5]}, 'xaxis': {'title': 'Brand'}}
        ]
    ),
    dict(
        label='Recommendation Score by Brand',
        method='update',
        args=[
            {'visible': [False, False, False, True]},
            {'title': 'Recommendation Score by Brand', 'yaxis': {'title': 'Recommendation Score'}, 'xaxis': {'title': 'Recommended Brand'}}
        ]
    )
]

fig.update_layout(
    title='Revenue by Brand',
    template='plotly_white',
    height=500,
    width=900,
    showlegend=False,
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        showactive=True,
        x=0.02,
        xanchor='left',
        y=1.20,
        yanchor='top'
    )],
    annotations=[dict(
        text='Select dashboard view:',
        x=0.02,
        xref='paper',
        y=1.28,
        yref='paper',
        showarrow=False,
        align='left'
    )],
    margin=dict(t=120, b=60, l=70, r=30),
    xaxis_title='Brand',
    yaxis_title='Revenue ($)'
)

fig.show()


# Simple KPI summary table

kpi_summary = pd.DataFrame({
    'Metric': [
        'Total Revenue',
        'Average Transaction Value',
        'Total Orders',
        'Brands Purchased',
        'Recommendation Rows'
    ],
    'Value': [
        f"${cleaned['line_total'].sum():,.2f}",
        f"${cleaned['line_total'].mean():,.2f}",
        f"{cleaned['order_id'].nunique():,}",
        f"{cleaned['brand'].nunique():,}",
        f"{len(rec_data):,}"
    ]
})

display(HTML('<h3>Dashboard KPI Summary</h3>'))
display(kpi_summary)

,Metric,Value
0,Total Revenue,"$83,383.50"
1,Average Transaction Value,$254.22
2,Total Orders,164
3,Brands Purchased,13
4,Recommendation Rows,88


Google. (n.d.). Gemini. Google. https://gemini.google.com/ I used Artificial Intelligience for assistance in creating the dropdown and selections.

In [ ]:
# keep in mind to close the connection
db_engine.dispose()

print ("Connection closed.")

Connection closed.
